# HCGC Colab Sweep Runner

Use one Colab session per dataset. Change `DATASET` to `imdb`, `dblp`, or `acm`, then run all cells.
Each session writes to a unique timestamped folder under `DRIVE_DIR`, so parallel sessions do not overwrite each other.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hcgc_results'

# Change this per Colab session: 'imdb', 'dblp', or 'acm'.
DATASET = 'imdb'

# Paper/presentation sweep settings.
MODELS = ['sage', 'rgcn', 'gat', 'appnp']
COMPRESSORS = ['hcgc', 'freehgc', 'cgc_homo', 'random_type', 'ahugc_style']
RATIOS = [0.5, 0.3, 0.25, 0.2, 0.15, 0.1]

# Fast first pass: RUNS=1, WARMUP=0. More stable paper run: RUNS=3, WARMUP=1.
RUNS = 1
WARMUP = 0
TRAIN_EPOCHS = 200
PRETRAIN_EPOCHS = 100
PRETRAIN_PATIENCE = 5
DEVICE = 'cuda'

# Keep False for the main HCGC result. Set True only for a quick smoke run.
NO_PRETRAIN = False
EMB_METHOD = 'gnn'
RATIO_SEARCH = 'fast'

# Optional label for the output folder.
JOB_NAME = DATASET

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

REPO_URL = 'https://github.com/ksj64381/HCGC.git'
WORKDIR = Path('/content/HCGC')
DATA_ROOT = Path(f'/content/hcgc_data_{DATASET}')

def run(cmd, cwd=None, check=True):
    print('$', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check)

Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

if (WORKDIR / '.git').exists():
    run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'])
else:
    run(['git', 'clone', REPO_URL, str(WORKDIR)])

run(['git', '-C', str(WORKDIR), 'rev-parse', '--short', 'HEAD'])

In [ ]:
import torch

print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cpu'
if torch.version.cuda:
    cuda_tag = 'cu' + torch.version.cuda.replace('.', '')
pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'

# Core packages. Colab already has torch; this installs PyG, dataset helpers, and build tools.
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'pybind11', 'scipy', 'pandas', 'ogb', 'matplotlib', 'tqdm'])

# Optional PyG sampling/scatter wheels. If a wheel is unavailable, full-batch IMDB/DBLP/ACM can still often run.
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv', '-f', pyg_wheel_url], check=False)

# Build the HCGC C++ kernel for the current Colab Linux/Python environment.
run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=str(WORKDIR))

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = Path(DRIVE_DIR) / 'by_dataset' / f'{JOB_NAME}_{timestamp}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / 'run.log'

cmd = [
    sys.executable, 'experiments.py',
    '--dataset', DATASET,
    '--models', *MODELS,
    '--compressors', *COMPRESSORS,
    '--ratios', *[str(r) for r in RATIOS],
    '--runs', str(RUNS),
    '--warmup', str(WARMUP),
    '--device', DEVICE,
    '--train-epochs', str(TRAIN_EPOCHS),
    '--pretrain-epochs', str(PRETRAIN_EPOCHS),
    '--pretrain-patience', str(PRETRAIN_PATIENCE),
    '--emb-method', EMB_METHOD,
    '--ratio-search', RATIO_SEARCH,
    '--root', str(DATA_ROOT),
    '--plot-dir', str(OUT_DIR),
]
if NO_PRETRAIN:
    cmd.append('--no-pretrain')

print('Output directory:', OUT_DIR)
print('Log:', LOG_PATH)
print('$', ' '.join(map(str, cmd)))

with LOG_PATH.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        cmd,
        cwd=str(WORKDIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f'experiments.py failed with exit code {code}')

(OUT_DIR / 'DONE.txt').write_text(
    f'dataset={DATASET}\nmodels={MODELS}\ncompressors={COMPRESSORS}\nratios={RATIOS}\n',
    encoding='utf-8',
)
print('Done:', OUT_DIR)

In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob(str(OUT_DIR / '*.csv')))
print('\n'.join(csv_files))

compare_csvs = [p for p in csv_files if 'compressor_compare' in Path(p).name]
if compare_csvs:
    df = pd.read_csv(compare_csvs[-1])
    display_cols = [
        'dataset', 'model', 'compressor', 'ratio',
        'node_compression', 'edge_compression', 'storage_compression',
        'test_acc_mean', 'test_macro_f1_mean', 't_total',
    ]
    display(df[[c for c in display_cols if c in df.columns]].head(20))